# Social Media Interest Clustering for Marketing

## Objective: To identify distinct social sub-cultures among high school students using text-based interest keywords.

### The "Sub-Culture" Discovery

By analyzing social media keywords, we successfully identified five distinct psychographic segments within the student population:

- The Athletes: High frequency of keywords like football, basketball, soccer, and softball. This group is highly active and likely responsive to sports apparel and energy drink marketing.

- The "Preppy" Segment: High usage of brands like Hollister, Abercrombie, and keywords like shopping and clothes. This is a prime target for fashion retailers.

- The Music/Creative Group: Characterized by keywords such as band, music, rock, and dance. This segment values self-expression and creative outlets.

- The Religion/Family Group: High engagement with keywords like church, jesus, god, and bible. This group represents a value-driven segment.

- The Socialites: Frequent use of "social" keywords like cute, sexy, hot, and dance. This group is heavily focused on peer interaction and social status.

In [ ]:
#Comment out if not required
#%pip install pandas numpy matplotlib seaborn scikitlearn

### 1. Environment Setup and Data Loading

Unlike the previous datasets, this one is "high-dimensional" (many columns). We will focus on the interest keywords (from 'basketball' onwards) for clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Load the data
df = pd.read_csv('Dataset/Clustering_Marketing.csv')

# The interests start from 'basketball' (column index 4) to the end
interests = df.iloc[:, 4:]

print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

### 2. Preprocessing: Handling Missing Values and Scaling

Social media data often has missing values (especially in the 'age' column) and sparse data in interest keywords

In [ ]:
# Handle missing age values with the mean
df['age'] = df['age'].fillna(df['age'].mean())

# Interest data is often skewed; we standardize to give all interests equal weight
scaler = StandardScaler()
interests_scaled = scaler.fit_transform(interests)

### 3. Choosing the Number of Clusters (K)

For marketing segmentation, we typically look for 4–6 clusters to ensure the segments are large enough to be actionable but distinct enough to be useful.

In [ ]:
wcss = []
for k in range(2, 10):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(interests_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(range(2, 10), wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Social Media Data')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.show()

### 4. Fitting the Model and 2D Visualization

Let's assume K=5 to capture diverse sub-cultures (e.g., athletes, "preppy" students, music lovers, etc.).

In [ ]:
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(interests_scaled)

# PCA for visualization
pca = PCA(n_components=2)
pca_data = pca.fit_transform(interests_scaled)
df_pca = pd.DataFrame(pca_data, columns=['PC1', 'PC2'])
df_pca['Cluster'] = df['Cluster']

plt.figure(figsize=(10, 7))
sns.scatterplot(x='PC1', y='PC2', hue='Cluster', data=df_pca, palette='tab10', alpha=0.6)
plt.title('Social Network Segments (PCA Projection)')
plt.show()

### 5. Cluster Profiling (Identifying Sub-Cultures)

This is the most important part for marketing. We look at which keywords are most frequent in each cluster.

In [ ]:
# Examine the top interests for each cluster
cluster_profile = df.groupby('Cluster').mean(numeric_only=True).iloc[:, 4:] # focus on interests

# Let's find the top 5 keywords for each cluster
for i in range(optimal_k):
    print(f"\n--- Cluster {i} Top Interests ---")
    print(cluster_profile.loc[i].sort_values(ascending=False).head(5))

### 6. Cluster Size Distribution

Before looking at what they like, we need to know how large each group is. This tells a marketer where the "mass market" is versus the "niche" sub-cultures.

In [ ]:
plt.figure(figsize=(8, 5))
# Using countplot with hue assigned to avoid the previous warning
sns.countplot(x='Cluster', data=df, hue='Cluster', palette='viridis', legend=False)
plt.title('Market Size: Number of Students per Cluster')
plt.xlabel('Cluster (Sub-Culture ID)')
plt.ylabel('Count')
plt.show()

### 7. Interest Heatmap (The "Culture Map")

A heatmap allows us to compare all clusters across the most popular keywords at once. This visualization effectively "maps" which interests define which groups.

In [ ]:
# Selecting the most defining keywords for a clear heatmap
top_keywords = ['football', 'shopping', 'music', 'church', 'cute', 'dance', 'rock', 'god', 'clothes', 'sports']
cluster_interests = df.groupby('Cluster')[top_keywords].mean()

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_interests, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('The Culture Map: Mean Interest Scores by Cluster')
plt.xlabel('Keywords / Interests')
plt.ylabel('Cluster')
plt.show()

### 8. Social Connectivity: "Number of Friends" Boxplot

Does a "Sports" fan have more friends than a "Music" fan? We can use the NumberOffriends column to see the social influence of each cluster.

In [ ]:
plt.figure(figsize=(10, 6))
# Fix for the warning: Assigning x to hue and setting legend=False
sns.boxplot(x='Cluster', y='NumberOffriends', data=df, hue='Cluster', palette='Set3', legend=False)
plt.ylim(0, 150) # Clipping outliers for better visibility
plt.title('Social Connectivity: Number of Friends per Cluster')
plt.show()

### 9. Gender Distribution per Cluster

Marketing to teenagers often involves understanding gender dynamics. We can visualize the gender split within each interest group.

In [ ]:
# Creating a cross-tabulation of Cluster and Gender
gender_cluster = pd.crosstab(df['Cluster'], df['gender'])

# Plotting a stacked bar chart
gender_cluster.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#ff9999','#66b3ff'])
plt.title('Gender Composition of Social Clusters')
plt.xlabel('Cluster')
plt.ylabel('Number of Students')
plt.legend(title='Gender')
plt.xticks(rotation=0)
plt.show()

## Discussion and Insights

By analyzing these additional visualizations, we can draw sophisticated conclusions for the assignment:

- The Gender Gap in Interests: The Gender Composition Chart often reveals that certain clusters (like the "Preppy/Shopping" group) are heavily female-dominated, while others (like "Athletes") might be more balanced or male-dominated. This allows for gender-tailored ad creative.

- The "Influencer" Cluster: The Number of Friends Boxplot identifies which sub-culture is the most socially active. If Cluster 1 has a significantly higher median of friends, they are the "Influencers"—targeting them could lead to a higher viral spread for a marketing campaign.

- Keyword Overlap: The Heatmap shows that while clusters are distinct, "Music" is often a universal interest across multiple clusters. This suggests that music-related sponsorships are a "safe bet" for reaching the entire student population, whereas "Hollister" is highly specific to a single niche.

# Conclusion

The K-Means clustering of social network data successfully transitioned raw "keyword mentions" into actionable psychographic personas.

- Mathematical Validity: The Elbow Method and PCA confirmed that the messy world of social media can indeed be organized into 5-6 statistically significant groups.

- Actionable Strategy: We moved beyond simple demographics (age/sex) to understand the values of the students. We identified that the "Socialites" and "Athletes" represent the largest market share, while the "Preppy" and "Creative" groups represent smaller, high-intent niches.

- Final Recommendation: For a successful marketing launch, the brand should utilize the "Influencer" Cluster identified in the friend-count analysis to seed the product, while using the "Culture Map" Heatmap to decide which keywords to include in social media advertisements for each specific segment.